# Prompt Management Architecture — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/llm-engineering/4-2/prompt-management-architecture.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/llm-engineering/4-2/prompt-management-architecture.ipynb)

Companion to: [Prompt Management Architecture: Prompts as Files, Not Strings](https://anacodicai.org/learn/llm-engineering/4-2/prompt-management-architecture)

*(The article's version uses Jinja2 for templating; this notebook uses a tiny
stdlib-only templater so it needs zero installs — the lesson is identical:
static prompts get cached, templates get re-rendered.)*


## What you'll build

1. Prompts stored as separate **files** (not string literals in code).
2. A `PromptManager` that **caches** static system prompts but **never** caches
   rendered templates.
3. A **provider-aware lookup** that selects the right system prompt file by
   provider, not by scattered `if/elif`.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
import tempfile
from pathlib import Path
from functools import lru_cache
print("ready — standard library only")

ready — standard library only


## Step 1 — prompts as files on disk

Real projects keep prompts in their own files, never as strings embedded in
application code. We build a tiny directory of them here so the rest of the
notebook is working with genuine files, not in-memory strings pretending to be
files.

In [2]:
def build_prompt_directory():
    tmp = Path(tempfile.mkdtemp())
    (tmp / "reviewer_system_default.txt").write_text(
        "You are a meticulous code reviewer. Be specific and cite line "
        "numbers. Never invent a bug that isn't present in the diff."
    )
    (tmp / "reviewer_system_compact.txt").write_text(
        "You are a code reviewer. Provider has a small context budget: be "
        "terse, list only the top three issues, no preamble."
    )
    return tmp

prompt_dir = build_prompt_directory()
print("prompt files on disk:", [p.name for p in prompt_dir.iterdir()])

prompt files on disk: ['reviewer_system_compact.txt', 'reviewer_system_default.txt']


## Step 2 — provider-aware selection: a lookup, not a conditional

Adding a new provider means adding one entry to this dictionary — not finding
and updating every `if provider == ...` scattered through the codebase.

In [3]:
PROVIDER_SYSTEM_PROMPT = {
    "default-tier": "reviewer_system_default",
    "small-context-tier": "reviewer_system_compact",
}
print(PROVIDER_SYSTEM_PROMPT)

{'default-tier': 'reviewer_system_default', 'small-context-tier': 'reviewer_system_compact'}


## Step 3 — the manager: cache static reads, never cache renders

`get()` reads a static file and caches it — the content never changes, so the
second read would return the identical string anyway. `render()` fills in a
tiny stdlib-only template and is **never** cached, because the variables are
different on every call.

In [4]:
class PromptManager:
    def __init__(self, prompt_dir):
        self.prompt_dir = Path(prompt_dir)
        self.read_count = {"get": 0, "render": 0}

    @lru_cache(maxsize=None)
    def _read_static(self, key):
        self.read_count["get"] += 1
        return (self.prompt_dir / f"{key}.txt").read_text()

    def get(self, key):
        return self._read_static(key)

    def render_review_request(self, language, focus, diff, prior_findings=""):
        # a minimal, stdlib-only "template": build the string in Python
        # instead of a template-language conditional block
        self.read_count["render"] += 1
        parts = [f"Review this {language} change for {focus}."]
        if prior_findings:
            parts.append(f"Known recurring issues in this codebase: {prior_findings}")
        parts.append(f"Diff:\n{diff}")
        return "\n\n".join(parts)

    def system_prompt_for_provider(self, provider):
        key = PROVIDER_SYSTEM_PROMPT[provider]
        return self.get(key)

pm = PromptManager(prompt_dir)
print("PromptManager ready")

PromptManager ready


## Step 4 — caching in action

**Try it:** call `system_prompt_for_provider` a few more times for either
provider and re-run — `read_count['get']` should not move.

In [5]:
for provider in PROVIDER_SYSTEM_PROMPT:
    text = pm.system_prompt_for_provider(provider)
    print(f"{provider:20} -> {text[:50]}...")

print(f"\nfile reads for get(): {pm.read_count['get']} "
      f"(one per distinct key, even if requested repeatedly)")

for _ in range(3):
    pm.system_prompt_for_provider("default-tier")
print(f"after 3 more repeat requests, file reads for get(): "
      f"{pm.read_count['get']} (unchanged -- served from cache)")

default-tier         -> You are a meticulous code reviewer. Be specific an...
small-context-tier   -> You are a code reviewer. Provider has a small cont...

file reads for get(): 2 (one per distinct key, even if requested repeatedly)
after 3 more repeat requests, file reads for get(): 2 (unchanged -- served from cache)


## Step 5 — the same template, different variables, different output

**Try it:** change `focus` or `diff` below and re-run — the assertion checks
that changing the variables genuinely changes the output, which is exactly
why templates must never be cached by their rendered result.

In [6]:
rendered_1 = pm.render_review_request(
    language="Python", focus="security", diff="+ eval(user_input)",
)
rendered_2 = pm.render_review_request(
    language="Python", focus="security", diff="+ eval(user_input)",
    prior_findings="unsanitized eval() calls (flagged 3 times before)",
)

print("--- call 1 (no prior findings) ---")
print(rendered_1)
print("\n--- call 2 (same diff, WITH prior findings) ---")
print(rendered_2)

assert rendered_1 != rendered_2
assert "Known recurring issues" not in rendered_1
assert "Known recurring issues" in rendered_2
print(f"\nasserts passed: identical diff, different variables, genuinely "
      f"different output")
print(f"template renders (never cached): {pm.read_count['render']}")

--- call 1 (no prior findings) ---
Review this Python change for security.

Diff:
+ eval(user_input)

--- call 2 (same diff, WITH prior findings) ---
Review this Python change for security.

Known recurring issues in this codebase: unsanitized eval() calls (flagged 3 times before)

Diff:
+ eval(user_input)

asserts passed: identical diff, different variables, genuinely different output
template renders (never cached): 2


## Try it yourself

- Add a third provider tier and its own system-prompt file — confirm you only
  had to touch the `PROVIDER_SYSTEM_PROMPT` dict, nothing else.
- Try "caching" `render_review_request`'s output by key alone (ignoring the
  variables) and see how it breaks — call 2 would silently return call 1's
  answer, minus the prior-findings context it needed.